This is the code that does all the preprocessing for the lasc expression file.
Requires the lasc expression file and the gene info annotation file as inputs. It also requires the main Orthogroup tsv file. It outputs two expression files, summed_lasc_expression and max_lasc_expression which are tsv, TPM normalized, and display orthogroups instead of genes.

In [ ]:
# Borderline esoteric code that adds the corresponding transcript length
# to each gene in the expression matrix, and then uses it to do TPM normalization

import pandas as pd

# Load the CSV files
expression_file = "/content/drive/MyDrive/expression/LS.expression.csv"
annotation_file = "/content/drive/MyDrive/expression/LSC_gene_info.csv"

# Read the expression and annotation data
expression_df = pd.read_csv(expression_file)
annotation_df = pd.read_csv(annotation_file)

# Merge transcript lengths into the expression data
expression_df = expression_df.merge(
    annotation_df[[annotation_df.columns[0], annotation_df.columns[5]]],
    left_on=expression_df.columns[0],
    right_on=annotation_df.columns[0],
    how="left"
)
expression_df.drop(columns=[annotation_df.columns[0]], inplace=True)
# Rename the new column for clarity
expression_df.rename(columns={annotation_df.columns[5]: "TranscriptLength"}, inplace=True)

# Ensure the TranscriptLength column is numeric
expression_df["TranscriptLength"] = pd.to_numeric(expression_df["TranscriptLength"], errors="coerce")

# Check for missing transcript lengths
if expression_df["TranscriptLength"].isnull().any():
    print("Warning: Some genes are missing transcript lengths!")
    # Drop rows with missing transcript lengths
    expression_df = expression_df.dropna(subset=["TranscriptLength"])

# Ensure all expression columns are numeric
expression_columns = expression_df.columns[1:-1]  # Skip the first (GeneName) and last (TranscriptLength) columns
for col in expression_columns:
    expression_df[col] = pd.to_numeric(expression_df[col], errors="coerce")

# Replace the expression columns with TPM-normalized value
# Step 1: Calculate the normalized value for each gene
for col in expression_columns:
    expression_df[col] = expression_df[col] / expression_df["TranscriptLength"]

# Step 2: Compute the scaling factor for each column and normalize to TPM
for col in expression_columns:
    scaling_factor = expression_df[col].sum()
    expression_df[col] = (expression_df[col] / scaling_factor) * 1e6

# Remove the TranscriptLength column as it is stupid
expression_df.drop(columns=["TranscriptLength"], inplace=True)

# Save the updated expression file with TPM values
output_file = "/content/drive/MyDrive/expression/LS.expression_norm.csv"
expression_df.to_csv(output_file, index=False)

print(f"Normalized expression data with TPM saved to {output_file}")


In [ ]:
# Same algorithm as before to create a translation file for the LASC OG
import csv
file_path_1 = "/content/drive/MyDrive/expression/LSC_gene_info.csv"
file_path_2 = "/content/drive/MyDrive/expression/Orthogroups.tsv"

# Load the mapping from the first file (augustus -> lascID)
format1_to_format2 = {}
with open(file_path_1, 'r') as file1:
    reader = csv.reader(file1)
    for row in reader:
        format1_to_format2[row[0]] = row[2]

# Load the mapping from the second file (lascID -> orthogroup)
format2_to_orthogroup = {}
with open(file_path_2, 'r') as file2:
    reader = csv.reader(file2, delimiter='\t')
    for row in reader:
        orthogroup = row[0]
        proteins_format2 = row[-1].split(', ')
        for protein in proteins_format2:
            format2_to_orthogroup[protein] = orthogroup

# Create the output (augustus -> orthogroup)
output_rows = []
for protein_format1, protein_format2 in format1_to_format2.items():
    orthogroup = format2_to_orthogroup.get(protein_format2)  # None if not found
    if orthogroup:  # Skip if orthogroup is None
        output_rows.append([protein_format1, orthogroup])

# Write the output to a new TSV file
with open('/content/drive/MyDrive/expression/lasc_to_OG.tsv', 'w', newline='') as output_file:
    writer = csv.writer(output_file, delimiter='\t')
    writer.writerows(output_rows)

In [ ]:
# Uses the lasc to OG translation file to rename the genes in the normalized
# expression file
import csv

# Load the mapping from the output TSV file (augustus -> orthogroup)
format1_to_orthogroup = {}
with open('/content/drive/MyDrive/expression/lasc_to_OG.tsv', 'r') as output_file:
    reader = csv.reader(output_file, delimiter='\t')
    for row in reader:
        format1_to_orthogroup[row[0]] = row[1]

# Process the input .txt file and replace the first column
updated_rows = []
with open('/content/drive/MyDrive/expression/LS.expression_norm.csv', 'r') as input_file:
    reader = csv.reader(input_file)
    header = next(reader)  # Skip the first row (header)
    for row in reader:
        format1_name = row[0]
        orthogroup = format1_to_orthogroup.get(format1_name, "NA")  # Use "NA" if no match
        updated_row = [orthogroup] + row[1:]  # Replace first column
        if orthogroup != "NA":
          updated_rows.append(updated_row)
updated_rows.insert(0, header)

# Write the modified data to a new file
with open('/content/drive/MyDrive/expression/lasc.expressionOG_norm.csv', 'w', newline='') as output_file:
    writer = csv.writer(output_file)
    writer.writerows(updated_rows)

print("Updated file written")

In [ ]:
# Uses pandas to filter the bridge expression file containing orthogroups
# so that only one entry per orthogroup remains, filtering can be altered.
# This creates 2 filtered files with summed orthogroup genes & the max values

import pandas as pd

# Load the data into a Pandas DataFrame
df = pd.read_csv('/content/drive/MyDrive/expression/lasc.expression_mean.csv')

# Function to select how the composite orthogroup entry is to be constructed,
def select_sums(group):
    return group.sum(numeric_only=True)  # Sums

def select_max(group):
    return group.max(numeric_only=True)  # Max

# Apply grouping and sum
merged_df = df.groupby('Geneid', as_index=False).apply(select_sums)

# Write the merged data to a new file
merged_df.to_csv('/content/drive/MyDrive/expression/summed_lasc.expression.txt', sep='\t', index=False)

# Apply grouping and select maximum values
merged_df = df.groupby('Geneid', as_index=False).apply(select_max)

# Write the merged data to a new file
merged_df.to_csv('/content/drive/MyDrive/expression/max_lasc.expression.txt', sep='\t', index=False)